Christian Thede
CNN Model

Notes: 
1. Used AI for mounting and dealing with colab and google drive (and still had a ton of issues)
2. 

In [ ]:
# import ml libraries
import tensorflow as tf  # noqa: F401
import skimage  # noqa: F401
# or ------
#import sys
#!{sys.executable} -m pip install -q tensorflow scikit-image scikit-learn matplotlib

# Google Drive mount for Colab
import os
import shutil

mountpoint = "/content/drive"

from google.colab import drive
drive.mount(mountpoint, force_remount=True)

In [ ]:
from pathlib import Path

# split
TRAIN_RATIO = 0.60
TEST_RATIO = 0.30
VAL_RATIO = 0.10

# Hardcoded Colab path options (case/folder variants)
NUMS_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/Nums"),
    Path("/content/drive/MyDrive/Colab Notebooks/nums"),
    Path("/content/drive/MyDrive/486-Final-Project/Nums"),
    Path("/content/drive/MyDrive/486-Final-Project/nums"),
]

NUMS_DIR = next((p for p in NUMS_CANDIDATES if p.exists()), NUMS_CANDIDATES[0])
OUTPUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Output")

print("NUMS_DIR:", NUMS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

if not NUMS_DIR.exists():
    raise FileNotFoundError(
        "Could not find the dataset folder in any expected location. "
        f"Checked: {[str(p) for p in NUMS_CANDIDATES]}"
    )

# Create output folder only after we confirm Drive + dataset path are valid.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split

from skimage import io, color
from skimage.transform import resize

### Data

In [ ]:
# local nums
IMAGE_SIZE = (28, 28)
NUM_CLASSES = 10

# Recursive search so subfolders are also included
image_paths = sorted(list(NUMS_DIR.rglob("*.png")) + list(NUMS_DIR.rglob("*.PNG")))
print("Found", len(image_paths), "image files")

if len(image_paths) == 0:
    raise FileNotFoundError(
        f"No PNG files found in {NUMS_DIR}. "
        "Check the folder path and run Cell 1 again (and mount Drive in Colab if needed)."
    )

X_list = []
y_list = []
skipped = []

for p in image_paths:
    name = p.stem
    if not name or not name[0].isdigit():
        skipped.append(p.name)
        continue

    label = int(name[0])
    img = io.imread(str(p))

    # make grayscale
    if img.ndim == 3:
        img = color.rgb2gray(img[..., :3])

    img = resize(img, IMAGE_SIZE, anti_aliasing=True)
    X_list.append(img[..., None])
    y_list.append(label)

if len(X_list) == 0:
    preview = image_paths[:5]
    raise ValueError("ValueError")

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)


# prints --------------------------------
print("Loaded", len(X), "usable images")
print("Skipped (no leading digit):", len(skipped))
print("X shape:", X.shape)
print("y labels:", sorted(set(y.tolist())))
# prints --------------------------------

# split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, train_size=TRAIN_RATIO, random_state=42, stratify=y
    )

# split into test and val
temp_test_ratio = TEST_RATIO / (TEST_RATIO + VAL_RATIO)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, train_size=temp_test_ratio, random_state=42
    )

y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_oh = keras.utils.to_categorical(y_test, NUM_CLASSES)
y_val_oh = keras.utils.to_categorical(y_val, NUM_CLASSES)

INPUT_CHANNELS = X_train.shape[-1]
print("Train:", X_train.shape, y_train_oh.shape)
print("Test:", X_test.shape, y_test_oh.shape)
print("Val:", X_val.shape, y_val_oh.shape)

### Model Building

In [ ]:
# build model
model = keras.Sequential(
    [
        layers.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], INPUT_CHANNELS)),
        layers.Conv2D(filters=5, kernel_size=3, padding="same", activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(filters=8, kernel_size=3, padding="same", activation="tanh"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(300, activation="tanh"),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ]
)

model.summary()

### Model Compilation

In [ ]:
model.compile(
optimizer=keras.optimizers.Adam(learning_rate=3e-4),
loss="categorical_crossentropy",
metrics=["accuracy"],
)

callbacks = [
keras.callbacks.EarlyStopping(
monitor="val_loss", patience=20, restore_best_weights=True
),
# stop for loss?
keras.callbacks.ReduceLROnPlateau(
monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6
),
]

### Model Training

In [ ]:
# train
history = model.fit(
    X_train,
    y_train_oh,
    epochs=100,
    batch_size=8,
    validation_data=(X_val, y_val_oh),
    callbacks=callbacks,
    verbose=1,
)

### Model Evaluation

In [ ]:
# evaluate
train_loss, train_acc = model.evaluate(X_train, y_train_oh, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val_oh, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test_oh, verbose=0)

print(f"Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.4f}")
print(f"Val loss:   {val_loss:.4f} | Val accuracy:   {val_acc:.4f}")
print(f"Test loss:  {test_loss:.4f} | Test accuracy:  {test_acc:.4f}")
print(f"Set sizes -> train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

# save model
model_path = OUTPUT_DIR / "digit_cnn.keras"
model.save(model_path)
print("Saved model to", model_path)

### Prediction

In [ ]:
# predictions
probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(probs, axis=1)

# sample views
for i in range(min(12, X_test.shape[0])):
    print(f"true={y_test[i]} pred={y_pred[i]} probs={np.round(probs[i], 3)}")

### Visualizations

In [ ]:
# visualizations
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.axhline(test_acc, linestyle="--", label=f"test ({test_acc:.2f})")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.axhline(test_loss, linestyle="--", label=f"test ({test_loss:.2f})")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# test images with predictions
n = min(12, X_test.shape[0])
plt.figure(figsize=(12, 3))
for i in range(n):
    ax = plt.subplot(2, (n + 1) // 2, i + 1)
    ax.imshow(X_test[i].squeeze(), cmap="gray")
    ax.set_title(f"t={y_test[i]} p={y_pred[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()